# v20.5 Regime Feature Importance (All Features)

This notebook follows the `v20.5` regime setup with all numeric features and exposes three different importance methods in separate cell blocks:
- XGBoost `feature_importances_`
- test-only permutation importance
- test-only SHAP importance


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import shap
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

SEED = 42
TARGET_COL = "soil_moisture_5cm"
T1 = 0.20
T2 = 0.313
TOP_K = 30
PERM_REPEATS = 5
EXCLUDE_COLS = {"station_id", "date", TARGET_COL}

def resolve_split_root() -> Path:
    candidates = [
        Path("Temporal/Pipeline/data/splits/derived_8.0"),
        Path("../../../../Temporal/Pipeline/data/splits/derived_8.0"),
    ]
    for candidate in candidates:
        resolved = candidate.resolve()
        if (resolved / "train.csv").exists():
            return resolved
    raise FileNotFoundError("Could not locate Temporal/Pipeline/data/splits/derived_8.0")

SPLIT_ROOT = resolve_split_root()
OUTPUT_DIR = Path("feature_importance_runs_full_v205")

XGB_PARAMS_DRY = dict(
    objective="reg:absoluteerror",
    random_state=SEED,
    n_jobs=-1,
    subsample=0.9,
    colsample_bytree=0.8,
    max_depth=8,
    min_child_weight=2,
    n_estimators=5500,
    learning_rate=0.04,
    reg_lambda=1.5,
    reg_alpha=0.03,
    gamma=0.0,
)

XGB_PARAMS_TRANSITION = dict(
    objective="reg:absoluteerror",
    random_state=SEED,
    n_jobs=-1,
    max_depth=7,
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.85,
    n_estimators=8000,
    learning_rate=0.03,
    reg_lambda=3.0,
    reg_alpha=0.05,
)

XGB_PARAMS_WET = dict(
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1,
    max_depth=10,
    min_child_weight=1,
    subsample=1.0,
    colsample_bytree=0.9,
    n_estimators=6000,
    learning_rate=0.03,
    reg_lambda=0.3,
    reg_alpha=0.0,
)

np.random.seed(SEED)
print("Split root:", SPLIT_ROOT)
print("Output dir:", OUTPUT_DIR.resolve())

In [ ]:
def load_splits(split_root: Path):
    train_path = split_root / "train.csv"
    val_path = split_root / "val.csv"
    test_path = split_root / "test.csv"
    for path in (train_path, val_path, test_path):
        if not path.exists():
            raise FileNotFoundError(f"Missing split file: {path}")
    return pd.read_csv(train_path), pd.read_csv(val_path), pd.read_csv(test_path)


def get_full_feature_cols(df: pd.DataFrame) -> list[str]:
    cols = []
    for col in df.columns:
        if col in EXCLUDE_COLS:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            cols.append(col)
    if not cols:
        raise ValueError("No numeric feature columns were found.")
    return cols


def clean_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    return df.replace([np.inf, -np.inf], np.nan)


def assign_regime(y: np.ndarray) -> np.ndarray:
    regime = np.zeros_like(y, dtype=int)
    regime[(y > T1) & (y <= T2)] = 1
    regime[y > T2] = 2
    return regime


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    bias = float(np.mean(y_pred - y_true))
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    ubrmse = float(np.sqrt(max(rmse ** 2 - bias ** 2, 0.0)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else float("nan")
    return {"R2": r2, "RMSE": rmse, "ubRMSE": ubrmse, "Bias": bias}


def builtin_importance_table(model, feature_cols):
    values = pd.Series(model.feature_importances_, index=feature_cols, dtype=float)
    total = float(values.sum())
    if total > 0:
        values = values / total
    return (
        values.rename("importance_mean")
        .reset_index()
        .rename(columns={"index": "feature"})
        .assign(importance_std=np.nan)
        .sort_values("importance_mean", ascending=False, kind="stable")
        .reset_index(drop=True)
    )


def permutation_importance_table(model, X, y, feature_cols, n_repeats=5, random_state=42):
    result = permutation_importance(
        model,
        X,
        y,
        scoring="r2",
        n_repeats=n_repeats,
        random_state=random_state,
        n_jobs=-1,
    )
    return (
        pd.DataFrame({
            "feature": feature_cols,
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std,
        })
        .sort_values("importance_mean", ascending=False, kind="stable")
        .reset_index(drop=True)
    )


def shap_importance_table(model, X, feature_cols):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    mean_abs = np.abs(shap_values).mean(axis=0)
    return (
        pd.DataFrame({
            "feature": feature_cols,
            "importance_mean": mean_abs,
            "importance_std": np.abs(shap_values).std(axis=0),
        })
        .sort_values("importance_mean", ascending=False, kind="stable")
        .reset_index(drop=True)
    )


train_df, val_df, test_df = load_splits(SPLIT_ROOT)
full_feature_cols = get_full_feature_cols(train_df)

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("Test rows:", len(test_df))
print("Full numeric feature count:", len(full_feature_cols))

In [ ]:
trainval_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)
y_trainval = trainval_df[TARGET_COL].to_numpy(dtype=float)
regime_trainval = assign_regime(y_trainval)

x_trainval_base = clean_numeric_df(trainval_df[full_feature_cols].copy())
x_test_base = clean_numeric_df(test_df[full_feature_cols].copy())
x_trainval_specialist = clean_numeric_df(trainval_df[full_feature_cols].copy())
x_test_specialist = clean_numeric_df(test_df[full_feature_cols].copy())

mask_base = regime_trainval == 0
mask_transition = regime_trainval == 1
mask_wet = regime_trainval == 2

y_test = test_df[TARGET_COL].to_numpy(dtype=float)
regime_test = assign_regime(y_test)
mask_base_test = regime_test == 0
mask_transition_test = regime_test == 1
mask_wet_test = regime_test == 2

print(
    f"Regime counts (train+val): dry={int(mask_base.sum())}, "
    f"transition={int(mask_transition.sum())}, wet={int(mask_wet.sum())}"
)
print(
    f"Regime counts (test): dry={int(mask_base_test.sum())}, "
    f"transition={int(mask_transition_test.sum())}, wet={int(mask_wet_test.sum())}"
)

In [ ]:
model_base = XGBRegressor(**XGB_PARAMS_DRY)
model_transition = XGBRegressor(**XGB_PARAMS_TRANSITION)
model_wet = XGBRegressor(**XGB_PARAMS_WET)

model_base.fit(x_trainval_base, y_trainval, verbose=0)
pred_base_test = model_base.predict(x_test_base)

x_trainval_aug = np.column_stack([
    x_trainval_specialist.to_numpy(dtype=float),
    model_base.predict(x_trainval_base),
])
x_test_aug = np.column_stack([
    x_test_specialist.to_numpy(dtype=float),
    pred_base_test,
])
aug_feature_cols = list(full_feature_cols) + ["base_pred"]

model_transition.fit(x_trainval_aug[mask_transition], y_trainval[mask_transition], verbose=0)
model_wet.fit(x_trainval_aug[mask_wet], y_trainval[mask_wet], verbose=0)

pred_transition_test = model_transition.predict(x_test_aug)
pred_wet_test = model_wet.predict(x_test_aug)

print("Models trained.")

In [ ]:
metrics_df = pd.DataFrame(
    {
        "dry": compute_metrics(y_test[mask_base_test], pred_base_test[mask_base_test]),
        "transition": compute_metrics(y_test[mask_transition_test], pred_transition_test[mask_transition_test]),
        "wet": compute_metrics(y_test[mask_wet_test], pred_wet_test[mask_wet_test]),
    }
).T[["R2", "RMSE", "ubRMSE", "Bias"]]
metrics_df.index.name = "regime"
metrics_df

## XGBoost `feature_importances_`

In [ ]:
xgb_importance_tables = {}
xgb_top_tables = {}

xgb_importance_tables["dry"] = builtin_importance_table(model_base, full_feature_cols)
xgb_top_tables["dry"] = xgb_importance_tables["dry"].head(TOP_K).copy()
xgb_top_tables["dry"]

In [ ]:
xgb_importance_tables["transition"] = builtin_importance_table(model_transition, aug_feature_cols)
xgb_top_tables["transition"] = xgb_importance_tables["transition"].head(TOP_K).copy()
xgb_top_tables["transition"]

In [ ]:
xgb_importance_tables["wet"] = builtin_importance_table(model_wet, aug_feature_cols)
xgb_top_tables["wet"] = xgb_importance_tables["wet"].head(TOP_K).copy()
xgb_top_tables["wet"]

## Test-Only Permutation Importance

In [ ]:
perm_importance_tables = {}
perm_top_tables = {}

perm_importance_tables["dry"] = permutation_importance_table(
    model_base,
    x_test_base.loc[mask_base_test],
    y_test[mask_base_test],
    full_feature_cols,
    n_repeats=PERM_REPEATS,
    random_state=SEED,
)
perm_top_tables["dry"] = perm_importance_tables["dry"].head(TOP_K).copy()
perm_top_tables["dry"]

In [ ]:
perm_importance_tables["transition"] = permutation_importance_table(
    model_transition,
    x_test_aug[mask_transition_test],
    y_test[mask_transition_test],
    aug_feature_cols,
    n_repeats=PERM_REPEATS,
    random_state=SEED,
)
perm_top_tables["transition"] = perm_importance_tables["transition"].head(TOP_K).copy()
perm_top_tables["transition"]

In [ ]:
perm_importance_tables["wet"] = permutation_importance_table(
    model_wet,
    x_test_aug[mask_wet_test],
    y_test[mask_wet_test],
    aug_feature_cols,
    n_repeats=PERM_REPEATS,
    random_state=SEED,
)
perm_top_tables["wet"] = perm_importance_tables["wet"].head(TOP_K).copy()
perm_top_tables["wet"]

## Test-Only SHAP Importance

In [ ]:
shap_importance_tables = {}
shap_top_tables = {}

shap_importance_tables["dry"] = shap_importance_table(
    model_base,
    x_test_base.loc[mask_base_test].to_numpy(dtype=float),
    full_feature_cols,
)
shap_top_tables["dry"] = shap_importance_tables["dry"].head(TOP_K).copy()
shap_top_tables["dry"]

In [ ]:
shap_importance_tables["transition"] = shap_importance_table(
    model_transition,
    x_test_aug[mask_transition_test],
    aug_feature_cols,
)
shap_top_tables["transition"] = shap_importance_tables["transition"].head(TOP_K).copy()
shap_top_tables["transition"]

In [ ]:
shap_importance_tables["wet"] = shap_importance_table(
    model_wet,
    x_test_aug[mask_wet_test],
    aug_feature_cols,
)
shap_top_tables["wet"] = shap_importance_tables["wet"].head(TOP_K).copy()
shap_top_tables["wet"]

## Ablation

In [ ]:
ablation_method = "perm"  # choose: "xgb", "perm", "shap"

method_top_tables = {
    "xgb": xgb_top_tables,
    "perm": perm_top_tables,
    "shap": shap_top_tables,
}

method_table_map = {
    "xgb": xgb_importance_tables,
    "perm": perm_importance_tables,
    "shap": shap_importance_tables,
}

selected_top_tables = method_top_tables[ablation_method]
selected_importance_tables = method_table_map[ablation_method]
print(f"Using top-30 features from: {ablation_method}")

In [ ]:
def get_regime_setup(regime_name):
    if regime_name == "dry":
        return {
            "params": XGB_PARAMS_DRY,
            "train_X": x_trainval_base,
            "train_y": y_trainval,
            "train_mask": mask_base,
            "test_X": x_test_base,
            "test_y": y_test,
            "test_mask": mask_base_test,
            "feature_cols": list(full_feature_cols),
        }
    if regime_name == "transition":
        return {
            "params": XGB_PARAMS_TRANSITION,
            "train_X": pd.DataFrame(x_trainval_aug, columns=aug_feature_cols),
            "train_y": y_trainval,
            "train_mask": mask_transition,
            "test_X": pd.DataFrame(x_test_aug, columns=aug_feature_cols),
            "test_y": y_test,
            "test_mask": mask_transition_test,
            "feature_cols": list(aug_feature_cols),
        }
    if regime_name == "wet":
        return {
            "params": XGB_PARAMS_WET,
            "train_X": pd.DataFrame(x_trainval_aug, columns=aug_feature_cols),
            "train_y": y_trainval,
            "train_mask": mask_wet,
            "test_X": pd.DataFrame(x_test_aug, columns=aug_feature_cols),
            "test_y": y_test,
            "test_mask": mask_wet_test,
            "feature_cols": list(aug_feature_cols),
        }
    raise ValueError(f"Unknown regime: {regime_name}")


def ablate_top_features(regime_name, top_df):
    setup = get_regime_setup(regime_name)
    baseline_metrics = metrics_df.loc[regime_name].to_dict()
    results = []

    for feature in top_df["feature"].tolist():
        keep_cols = [c for c in setup["feature_cols"] if c != feature]
        X_tr = setup["train_X"][keep_cols]
        X_te = setup["test_X"][keep_cols]
        y_tr = setup["train_y"][setup["train_mask"]]
        y_te = setup["test_y"][setup["test_mask"]]

        model = XGBRegressor(**setup["params"])
        model.fit(X_tr.loc[setup["train_mask"]], y_tr, verbose=0)
        pred = model.predict(X_te.loc[setup["test_mask"]])
        m = compute_metrics(y_te, pred)
        results.append({
            "feature": feature,
            "R2": m["R2"],
            "RMSE": m["RMSE"],
            "ubRMSE": m["ubRMSE"],
            "Bias": m["Bias"],
            "delta_R2": m["R2"] - baseline_metrics["R2"],
            "delta_RMSE": m["RMSE"] - baseline_metrics["RMSE"],
            "delta_ubRMSE": m["ubRMSE"] - baseline_metrics["ubRMSE"],
            "delta_Bias": m["Bias"] - baseline_metrics["Bias"],
        })

    out = pd.DataFrame(results).sort_values("delta_R2").reset_index(drop=True)
    return out


In [ ]:
ablation_results = {}
ablation_results["dry"] = ablate_top_features("dry", selected_top_tables["dry"])
ablation_results["dry"]

In [ ]:
ablation_results["transition"] = ablate_top_features("transition", selected_top_tables["transition"])
ablation_results["transition"]

In [ ]:
ablation_results["wet"] = ablate_top_features("wet", selected_top_tables["wet"])
ablation_results["wet"]

## Export

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def export_method_tables(method_name, tables, top_tables):
    for regime_name, df in tables.items():
        df.to_csv(OUTPUT_DIR / f"{method_name}_{regime_name}_importance_full.csv", index=False)
        top_tables[regime_name].to_csv(
            OUTPUT_DIR / f"{method_name}_{regime_name}_importance_top{TOP_K}.csv",
            index=False,
        )
    combined = pd.concat(
        [top_tables["dry"].assign(regime="dry"),
         top_tables["transition"].assign(regime="transition"),
         top_tables["wet"].assign(regime="wet")],
        ignore_index=True,
    )[["regime", "feature", "importance_mean", "importance_std"]]
    combined.to_csv(OUTPUT_DIR / f"{method_name}_regime_importance_top{TOP_K}_combined.csv", index=False)

export_method_tables("xgb", xgb_importance_tables, xgb_top_tables)
export_method_tables("perm", perm_importance_tables, perm_top_tables)
export_method_tables("shap", shap_importance_tables, shap_top_tables)
for regime_name, df in ablation_results.items():
    df.to_csv(OUTPUT_DIR / f"{ablation_method}_ablation_{regime_name}.csv", index=False)
metrics_df.to_csv(OUTPUT_DIR / "regime_metrics.csv")

print("Saved outputs to:", OUTPUT_DIR.resolve())
print(f"Permutation repeats per feature: {PERM_REPEATS}")
print(f"Ablation source method: {ablation_method}")